# GKC Mash: Loading and Shaping Source Data

This notebook demonstrates the mash module's role as GKC's read/retrieval layer. The mash module provides APIs for loading data from multiple sources (Wikibase, Wikipedia, etc.), shaping entities into templates, and extending the system with new source adapters.

Topics covered:
- `WikibaseApiClient` — Generic Wikibase API client
- `WikibaseLoader` — High-level Wikidata/Wikibase retrieval orchestrator
- `MashSourceAdapter` — Plugin protocol for custom source loaders
- Template filtering and export methods
- Batch loading and polymorphic dispatch

## Imports

Import the key mash classes and functions.

In [ ]:
from gkc.mash import (
    ClaimSummary,
    WikibaseApiClient,
    WikibaseLoader,
    WikibaseMashSourceAdapter,
    WikipediaLoader,
    WikipediaMashSourceAdapter,
    apply_item_property_filters,
    apply_template_language_filter,
    fetch_property_labels,
    strip_entity_identifiers,
)

## WikibaseApiClient: Generic Wikibase API Access

The `WikibaseApiClient` provides low-level HTTP access to any Wikibase-compatible API (Wikidata, Data Distillery, custom instances).

In [ ]:
# Connect to Wikidata public API
client = WikibaseApiClient(api_url="https://www.wikidata.org/w/api.php")

# Search for entities by label
results = client.search_entities(
    label="Cherokee Nation",
    entity_type="item",
    language="en",
    limit=5,
)

print(f"Found {len(results)} matches")
if results:
    print(f"Top result: {results[0]['id']} - {results[0]['label']}")

## Fetch Single Entity

Retrieve a single entity by ID.

In [ ]:
# Get Cherokee Nation (Q14708404)
entity = client.get_entity("Q14708404")

print(f"Entity ID: {entity['id']}")
print(f"Label (en): {entity.get('labels', {}).get('en', {}).get('value', 'N/A')}")
print(f"Number of claims: {len(entity.get('claims', {}))}")

## Fetch Multiple Entities in Batch

Retrieve multiple entities efficiently in a single request.

In [ ]:
# Get multiple items at once
batch = client.get_entities(["Q14708404", "Q1783171"])

print(f"Retrieved {len(batch)} entities")
for entity_id, entity_data in batch.items():
    label = entity_data.get('labels', {}).get('en', {}).get('value', 'N/A')
    print(f"  {entity_id}: {label}")

## Utility: Fetch Property Labels

Retrieve human-readable labels for Wikidata properties using SPARQL.

In [ ]:
labels = fetch_property_labels(["P31", "P279", "P1705"], language="en")

print("Property labels:")
for prop_id, label in labels.items():
    print(f"  {prop_id}: {label}")

## ClaimSummary: Simple Claim Representation

A simplified representation of a Wikidata claim for display and export.

In [ ]:
claim = ClaimSummary(
    property_id="P31",
    value="Q5",
    rank="normal",
    qualifiers=[{"property": "P580", "value": "2005-01-15"}],
    references=[{"P248": "Q123"}],
)

print(f"Property: {claim.property_id}")
print(f"Value: {claim.value}")
print(f"Rank: {claim.rank}")
print(f"Qualifiers: {len(claim.qualifiers)}")
print(f"References: {len(claim.references)}")

## WikibaseLoader: High-Level Entity Retrieval

Load Wikibase items, properties, and entity schemas as templates. WikibaseLoader handles the orchestration of API calls and template construction.

In [ ]:
loader = WikibaseLoader()

# Load an item
item = loader.load_item("Q14708404")
print(f"Loaded item: {item.qid}")
print(f"Labels: {len(item.labels)}")
print(f"Claims: {len(item.claims)}")

## WikibaseItemTemplate: Filtering and Transformation

Use explicit helper functions to filter templates (language, properties, qualifiers, references) and prepare them for export.

In [ ]:
# Load and filter
template = loader.load_item("Q14708404")

print(f"Original claim count: {len(template.claims)}")

# Filter to specific properties
apply_item_property_filters(template, include_properties=["P31", "P1705"])
print(f"After property filter: {len(template.claims)}")

# Filter to English only (labels, descriptions, aliases)
apply_template_language_filter(template, ["en"])

print(f"Languages after filter: {list(template.labels.keys())}")

## WikidataTemplate Export Formats

Export templates in different formats for various workflows.

In [ ]:
template = loader.load_item("Q14708404")

# Filter before export
apply_template_language_filter(template, ["en"])
apply_item_property_filters(template, include_properties=["P31", "P21", "P569"])

# Summary (for display)
summary = template.summary()
print(f"Summary keys: {list(summary.keys())}")

# Full dict (original entity JSON)
full = template.to_dict()
print(f"Full dict has 'id': {'id' in full}")

# Simple dict (labels + claims)
simple = template.to_simple_dict()
print(f"Simple dict keys: {list(simple.keys())}")

# Shell (stripped for new item)
shell = template.to_shell()
print(f"Shell has 'id': {'id' in shell}")

# QuickStatements V1 format
qsv1 = template.to_qsv1(for_new_item=False)
print(f"QSV1 output length: {len(qsv1)} chars")
print(f"QSV1 first line: {qsv1.split(chr(10))[0] if qsv1 else 'N/A'}")

## WikibasePropertyTemplate: Loading and Exporting Properties

Load Wikibase properties and export their metadata.

In [ ]:
prop = loader.load_property("P31")

print(f"Loaded property: {prop.pid}")
print(f"Labels: {len(prop.labels)}")

# Filter to English
apply_template_language_filter(prop, ["en"])
print(f"English label: {prop.labels.get('en', 'N/A')}")

# Summary and export
print(f"Summary keys: {list(prop.summary().keys())}")
print(f"Dict keys: {list(prop.to_dict().keys())}")
print(f"Shell has 'id': {'id' in prop.to_shell()}")

## WikidataLoader: Load Entity Schema Template

Load a Wikidata EntitySchema as a template.

In [ ]:
schema = loader.load_entity_schema("E502")

print(f"Loaded schema: {schema.eid}")
print(f"Labels: {len(schema.labels)}")

# Filter to English
apply_template_language_filter(schema, ["en"])
print(f"English label: {schema.labels.get('en', 'N/A')}")

# Summary and export
print(f"Summary keys: {list(schema.summary().keys())}")
print(f"Dict keys: {list(schema.to_dict().keys())}")
print(f"Shell has 'id': {'id' in schema.to_shell()}")

## WikipediaLoader and WikipediaTemplate: Wikipedia Templates

Load Wikipedia infobox and other templates from en.wikipedia.org.

In [ ]:
wiki_loader = WikipediaLoader()

template = wiki_loader.load_template("Infobox settlement")

print(f"Template name: {template.title}")
summary = template.summary()
print(f"Summary keys: {list(summary.keys())}")
print(f"Dict keys: {list(template.to_dict().keys())}")

## Next Steps

The mash module provides read/retrieval capabilities for:

1. **Generic Wikibase operations** via `WikibaseApiClient` — Low-level HTTP access
2. **High-level entity loading** via `WikibaseLoader` and `WikipediaLoader` — Orchestration and template construction
3. **Unified source adapters** via `MashSourceAdapter` protocol — Polymorphic loading from multiple sources
4. **Template filtering and export** — Language filtering, property filtering, format transforms

To extend mash with new sources (CSV, JSON APIs, databases, etc.), implement the `MashSourceAdapter` protocol in a new loader/adapter pair.

For write operations (creating/updating items), see the [shipper module](../docs/gkc/api/shipper.md).

For more details:
- [Mash Overview Documentation](../docs/gkc/mash.md)
- [Mash API Reference](../docs/gkc/api/mash.md)
- [Mash CLI Commands](../docs/gkc/cli/mash.md)

## MashSourceAdapter: Plugin Pattern for Multiple Sources

The `MashSourceAdapter` protocol defines a unified interface for loading data from any source (Wikibase, Wikipedia, CSV, JSON APIs, etc.). Both Wikibase and Wikipedia loaders wrap as adapters following this contract.

In [ ]:
# Wikibase adapter handles Q/P/E dispatch automatically
wikibase_adapter = WikibaseMashSourceAdapter()

# Load different entity types
item = wikibase_adapter.load("Q42")
prop = wikibase_adapter.load("P31")
schema = wikibase_adapter.load("E502")

print(f"Loaded {wikibase_adapter.source_name} item: {item.qid}")
print(f"Loaded {wikibase_adapter.source_name} property: {prop.pid}")
print(f"Loaded {wikibase_adapter.source_name} schema: {schema.eid}")

# Batch loading
batch = wikibase_adapter.load_many(["Q42", "Q5", "P31"])
print(f"Batch loaded {len(batch)} items via adapter")

In [ ]:
# Wikipedia adapter loads templates
wikipedia_adapter = WikipediaMashSourceAdapter()

# Both formats work (normalized internally)
template1 = wikipedia_adapter.load("Infobox_settlement")
template2 = wikipedia_adapter.load("Template:Infobox_settlement")

print(f"Loaded {wikipedia_adapter.source_name} template: {template1.title}")
print(f"Both references load same template: {template1.title == template2.title}")

# Check if adapter can load a reference
print(f"Can load 'Infobox_settlement': {wikipedia_adapter.can_load('Infobox_settlement')}")
print(f"Can load 'NonexistentTemplate': {wikipedia_adapter.can_load('NonexistentTemplate')}")